In [1]:
from sage.all import * 
from Crypto.Hash import SHAKE256

In [2]:
def int_to_4_u64(x: int):
    mask = (1 << 64) - 1
    return [
        (x >> (64 * 0)) & mask,
        (x >> (64 * 1)) & mask,
        (x >> (64 * 2)) & mask,
        (x >> (64 * 3)) & mask,
    ]

In [3]:
def u_64_chunks_to_int(chunks):
    x = 0
    for el in reversed(chunks):
        x *= 2**64
        x += el
    return x

In [4]:
def u8_to_u64(u8_list):
    assert len(u8_list) == 32
    
    result = []
    
    for i in range(0, 32, 8):
        value = 0
        for j in range(8):
            value |= u8_list[i + j] << (8 * j)  # little endian
        result.append(value)
        
    return result

In [5]:
q = 0x73eda753299d7d483339d80809a1d80553bda402fffe5bfeffffffff00000001
n = 4

rounds = 16

byte_length = rounds * n * ceil(log(q, 2**8))
byte_length

2048

In [6]:
g = 7
g_inv = xgcd(g, q)[1] % q

int_to_4_u64(g_inv)

[15811494919095339301,
 12265042158844910152,
 6325132277814092362,
 2386719102704128386]

In [7]:
int_to_4_u64(2 + g)

[9, 0, 0, 0]

In [8]:
int_to_4_u64(1 + g)

[8, 0, 0, 0]

In [9]:
int_to_4_u64(g**2 + 2 * g + 1)

[64, 0, 0, 0]

In [10]:
int_to_4_u64(g**2 + g + 1)

[57, 0, 0, 0]

In [11]:
seed = ("Anemoi_" + str(q) + "_n_" + str(n) + "_g_" + str(g) + "_rounds_" + str(rounds)).encode('ascii')
shake = SHAKE256.new()
shake.update(seed)

In [12]:
constants_raw = list(shake.read(byte_length))
constants = []

for r in range(0, rounds):
    cnst = []
    for i in range(0, n):
        c = u_64_chunks_to_int(u8_to_u64(constants_raw[(n * r + i) * ceil(log(q, 2**8)):(n * r + i + 1) * ceil(log(q, 2**8))]))
        c = c % q
        cnst.append(int_to_4_u64(c))
    constants.append(cnst)

In [14]:
print("[")
for cnst in constants:
    txt = "["
    for c in cnst:
        txt += str(c) + ","
    txt += "],"
    print(txt)
print("]")

[
[[15401157979506806243, 6510228464654367299, 12516273982820336922, 3667998198539878104],[1272938555936473880, 10968473431683535069, 1748139904836803321, 3827530625666699398],[2370918801150047768, 18375521257712826975, 5231653088424157831, 2493432084922449013],[8580698330717978716, 3520766567961097622, 1108667916230912375, 6171476630144282050],],
[[378453737975831390, 15480171932193534065, 2710223916559690614, 4093736732645900826],[17902870080751891884, 3429890304449745620, 16833121360094561974, 4113981413389890254],[773869934470199535, 10967627189223064328, 17307922679080034432, 5108038414181438763],[12541421731521597147, 12013866273972511644, 13381401472885967307, 1515692304420789166],],
[[8890646007880030454, 16097164852115618123, 57412801160625020, 7170214708889636564],[11305445125368958994, 10528314019897077499, 9676442026909469444, 5283456572860005104],[14818764684497274677, 15808313283801441308, 7350896078092148889, 887291816749391567],[12341284865083335438, 1729970299806710778